In [0]:
dbutils.fs.unmount("/mnt/weather_data")

In [0]:
# Define variables
storage_account_name = "aimasterdata"
storage_account_key = "j0/fRQr9nKT5rozOswNOMIOWQIwyudJH5oDSXINkttZBTP9T1O8phxCx3bJlPjz/kG9gz5Rqd4oW+AStbqPmlw=="
container_name = "weatherdata"
mount_point = "/mnt/weather_data"

# Mount the storage account
dbutils.fs.mount(
    source=f"wasbs://{container_name}@{storage_account_name}.blob.core.windows.net",
    mount_point=mount_point,
    extra_configs={f"fs.azure.account.key.{storage_account_name}.blob.core.windows.net": storage_account_key}
)

# Verify mount
display(dbutils.fs.ls(mount_point))


In [0]:
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("delimiter", ";") \
    .option("inferSchema", "true") \
    .load("/mnt/weather_data/TempEPrecip.csv")

df.display()


In [0]:
from pyspark.sql.functions import col, regexp_replace
import pandas as pd

# Define file path
file_path = "/mnt/weather_data/TempEPrecip.csv"

# Read the CSV with semicolon delimiter
df_raw = spark.read.format("csv") \
    .option("header", "false") \
    .option("delimiter", ";") \
    .load(file_path)

# Display raw data
df_raw.display()


In [0]:
# Ensure Date column is included
columns_to_keep = [0]  # Assuming Date is always the first column
cleaned_columns = ["Date"]  # First column must be "Date"

# Process remaining columns, skipping "FLAG"
for i in range(1, len(header_stations)):  # Start from index 1 (skip "Date" column)
    station = header_stations[i]
    variable = header_variables[i]

    if station and variable and "FLAG" not in variable:
        columns_to_keep.append(i)  # Store valid column index
        cleaned_columns.append(f"{station.strip()}_{variable.strip()}")  # Format column name

# Print debug info
print(f"Final columns_to_keep: {len(columns_to_keep)}")
print(f"Final cleaned_columns: {len(cleaned_columns)}")


In [0]:
print(f"Columns detected in dataset: {len(columns_to_keep)}")
print(f"Columns assigned for renaming: {len(cleaned_columns)}")

In [0]:
# Select only columns that are NOT "FLAG"
df_filtered = df_raw.select([col(f"_c{i}") for i in columns_to_keep])

# Rename columns properly
df_cleaned = df_filtered.toDF(*cleaned_columns)

df_cleaned.display()


In [0]:
# Extract station names by removing the measurement type (e.g., "_Temperature")
stations = list(set([col.split("_")[0] for col in cleaned_columns if "_" in col]))

# Print debug info
print(f"Total unique weather stations found: {len(stations)}")


In [0]:
dbutils.fs.rm("/mnt/weather_data/split_stations/", True)


In [0]:
from unidecode import unidecode
import re

# Define output directory in Databricks storage
output_dir = "/mnt/weather_data/split_stations/"

# Loop through each station and create separate CSV files
for station in stations:
    # Select only relevant columns (date + station-specific data)
    station_columns = ["Date"] + [col for col in df_cleaned.columns if col.startswith(station)]
    
    # Create a DataFrame for this station
    df_station = df_cleaned.select(*station_columns)

    # **Normalize the station name**
    station_clean = unidecode(station)  # Remove accents
    station_clean = re.sub(r"\s*\(.*?\)", "", station_clean)  # Remove anything inside parentheses
    station_clean = station_clean.replace(" ", "_")  # Replace spaces with underscores

    # Define output file path
    output_path = f"{output_dir}{station_clean}.csv"

    # Save to CSV (overwrite if it already exists)
    df_station.write.format("csv") \
        .mode("overwrite") \
        .option("header", "true") \
        .save(output_path)

    print(f"✅ Saved: {output_path}")


In [0]:
display(dbutils.fs.ls("/mnt/weather_data/split_stations/"))


In [0]:
# Example: Read one station's CSV file (replace with an actual filename from the previous step)
file_to_check = "/mnt/weather_data/split_stations/ALAGOA_(12G/05C).csv"

df_check = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(file_to_check)

# Display the file contents
df_check.display()


In [0]:
pip install unidecode

In [0]:
from unidecode import unidecode
import re

# Define output directory
output_dir = "/mnt/weather_data/split_stations/"

# Extract unique stations for humidity
stations = list(set([col.split("_")[0] for col in cleaned_columns if "_" in col]))

# Loop through each station and create separate CSVs
for station in stations:
    # Select relevant columns (date + station-specific data)
    station_columns = ["Date"] + [col for col in df_cleaned.columns if col.startswith(station)]
    
    # Create a DataFrame for this station
    df_station = df_cleaned.select(*station_columns)

    # **Normalize the station name**
    # Remove accents
    station_clean = unidecode(station)

    # Remove the last part inside parentheses
    station_clean = re.sub(r"\s*\(.*?\)", "", station_clean)

    # Replace spaces with underscores for filenames
    station_clean = station_clean.replace(" ", "_")

    # Define output file path
    output_path = f"{output_dir}{station_clean}.csv"

    # Save to CSV
    df_station.write.format("csv") \
        .mode("overwrite") \
        .option("header", "true") \
        .save(output_path)

    print(f"✅ Saved: {output_path}")


In [0]:
# Load raw humidity data
file_path_humidity = "/mnt/weather_data/humidade.csv"

df_raw_humidity = spark.read.format("csv") \
    .option("header", "false") \
    .option("delimiter", ";") \
    .load(file_path_humidity)

df_raw_humidity.display()


In [0]:
# Extract headers
header_stations_humidity = df_raw_humidity.collect()[2]  # Location names
header_variables_humidity = df_raw_humidity.collect()[3]  # Measurement type

# Remove "FLAG" columns and handle NoneType errors
columns_to_keep_humidity = [0]  # Always keep the "Date" column
cleaned_columns_humidity = ["Date"]

for i in range(1, len(header_stations_humidity)):  
    station = header_stations_humidity[i]
    variable = header_variables_humidity[i]

    if station and variable and "FLAG" not in variable:
        columns_to_keep_humidity.append(i)
        cleaned_columns_humidity.append(f"{station.strip()}_{variable.strip()}")

# Select only relevant columns
df_filtered_humidity = df_raw_humidity.select([col(f"_c{i}") for i in columns_to_keep_humidity])

# Rename columns
df_cleaned_humidity = df_filtered_humidity.toDF(*cleaned_columns_humidity)

df_cleaned_humidity.display()


In [0]:
# Load raw wind power data
file_path_wind = "/mnt/weather_data/vento.csv"

df_raw_wind = spark.read.format("csv") \
    .option("header", "false") \
    .option("delimiter", ";") \
    .load(file_path_wind)

df_raw_wind.display()


In [0]:
# Extract headers
header_stations_wind = df_raw_wind.collect()[2]  # Location names
header_variables_wind = df_raw_wind.collect()[3]  # Measurement type

# Remove "FLAG" columns and handle NoneType errors
columns_to_keep_wind = [0]  # Always keep the "Date" column
cleaned_columns_wind = ["Date"]

for i in range(1, len(header_stations_wind)):  
    station = header_stations_wind[i]
    variable = header_variables_wind[i]

    if station and variable and "FLAG" not in variable:
        columns_to_keep_wind.append(i)
        cleaned_columns_wind.append(f"{station.strip()}_{variable.strip()}")

# Select only relevant columns
df_filtered_wind = df_raw_wind.select([col(f"_c{i}") for i in columns_to_keep_wind])

# Rename columns
df_cleaned_wind = df_filtered_wind.toDF(*cleaned_columns_wind)

df_cleaned_wind.display()


In [0]:
# Define output directory for humidity
output_dir_humidity = "/mnt/weather_data/split_stations/humidity/"
output_dir_wind = "/mnt/weather_data/split_stations/wind/"

# Extract unique stations for humidity
stations_humidity = list(set([col.split("_")[0] for col in cleaned_columns_humidity if "_" in col]))

# Save humidity data
for station in stations_humidity:
    station_columns = ["Date"] + [col for col in df_cleaned_humidity.columns if col.startswith(station)]
    df_station = df_cleaned_humidity.select(*station_columns)

    # Normalize filename
    station_clean = unidecode(station)
    station_clean = re.sub(r"\s*\(.*?\)", "", station_clean).replace(" ", "_")
    output_path = f"{output_dir_humidity}{station_clean}.csv"

    df_station.write.format("csv").mode("overwrite").option("header", "true").save(output_path)
    print(f"✅ Humidity data saved: {output_path}")

# Extract unique stations for wind power
stations_wind = list(set([col.split("_")[0] for col in cleaned_columns_wind if "_" in col]))

# Save wind power data
for station in stations_wind:
    station_columns = ["Date"] + [col for col in df_cleaned_wind.columns if col.startswith(station)]
    df_station = df_cleaned_wind.select(*station_columns)

    # Normalize filename
    station_clean = unidecode(station)
    station_clean = re.sub(r"\s*\(.*?\)", "", station_clean).replace(" ", "_")
    output_path = f"{output_dir_wind}{station_clean}.csv"

    df_station.write.format("csv").mode("overwrite").option("header", "true").save(output_path)
    print(f"✅ Wind power data saved: {output_path}")


In [0]:
# Define the base directory
base_dir = "/mnt/weather_data/split_stations/"

# Normalize location names (remove accents, fix underscores, and remove extra parts)
def normalize_name(name):
    name_clean = unidecode(name)  # Remove accents
    name_clean = re.sub(r"\s*\(.*?\)", "", name_clean)  # Remove everything inside parentheses
    name_clean = name_clean.replace(" ", "_")  # Replace spaces with underscores
    return name_clean

# Normalize all location names
locations_normalized = set([normalize_name(loc) for loc in stations]) & \
                       set([normalize_name(loc) for loc in stations_humidity]) & \
                       set([normalize_name(loc) for loc in stations_wind])

print(f"✅ Using {len(locations_normalized)} locations after normalization.")


In [0]:
from pyspark.sql.functions import col

# Get a list of correct locations
locations = list(locations_normalized)

for location in locations:
    print(f"🔹 Merging data for {location}...")

    # Define the corrected file paths
    temp_file = f"{base_dir}{location}.csv"
    humidity_file = f"{base_dir}humidity/{location}.csv"
    wind_file = f"{base_dir}wind/{location}.csv"

    # Check if files exist before loading
    try:
        df_temp = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(temp_file)
        df_humidity = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(humidity_file)
        df_wind = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(wind_file)

        # Perform an **inner join** on "Date"
        df_merged = df_temp.join(df_humidity, ["Date"], "inner").join(df_wind, ["Date"], "inner")

        # Save merged data
        merged_output_path = f"{base_dir}merged/{location}.csv"
        df_merged.write.format("csv").mode("overwrite").option("header", "true").save(merged_output_path)

        print(f"✅ Merged data saved: {merged_output_path}")

    except Exception as e:
        print(f"❌ Skipping {location} due to missing files: {e}")


In [0]:
# dbutils.fs.rm("/mnt/weather_data/split_stations/merged/PROENCA_A_NOVA.csv", recurse=True)
# dbutils.fs.rm("/mnt/weather_data/split_stations/merged/SANTAREM.csv", recurse=True)
# dbutils.fs.rm("/mnt/weather_data/split_stations/merged/SAO_BRAS_DE_ALPORTEL.csv", recurse=True)
# dbutils.fs.rm("/mnt/weather_data/split_stations/merged/ALCACOVAS.csv", recurse=True)

In [0]:
dbutils.fs.mv("/mnt/weather_data/split_stations/merged/PROENA-A-NOVA.csv", "/mnt/weather_data/split_stations/merged/PROENCA_A_NOVA.csv", recurse=True)
dbutils.fs.mv("/mnt/weather_data/split_stations/merged/SANTARM.csv", "/mnt/weather_data/split_stations/merged/SANTAREM.csv", recurse=True)
dbutils.fs.mv("/mnt/weather_data/split_stations/merged/SO_BRS_DE_ALPORTEL.csv", "/mnt/weather_data/split_stations/merged/SAO_BRAS_DE_ALPORTEL.csv", recurse=True)
dbutils.fs.mv("/mnt/weather_data/split_stations/merged/ALCOVAS.csv", "/mnt/weather_data/split_stations/merged/ALCACOVAS.csv", recurse=True)


In [0]:
display(dbutils.fs.ls("/mnt/weather_data/split_stations/merged/"))

In [0]:
# Example: Read the "BATALHA.csv" file (replace with any filename you want to check)
file_to_check = "/mnt/weather_data/split_stations/merged/ALCACOVAS.csv"

df_check = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(file_to_check)

# Display the contents of the file
df_check.display()

In [0]:
from pyspark.sql.functions import col

# Define mapping for renaming columns
column_rename_map = {
    "Temperatura do ar m�dia di�ria (�C)": "Temperatura media do ar diaria (C)",
    "Precipita��o di�ria (mm)": "Precipitacao diaria (mm)",
    "Humidade relativa m�dia di�ria (%)": "Humidade relativa media diaria (%)",
    "Velocidade do vento m�dia di�ria (m/s)": "Velocidade do vento media diaria (m/s)"
}

# Define base directory
merged_dir = "/mnt/weather_data/split_stations/merged/"

# Get list of merged files
merged_files = dbutils.fs.ls(merged_dir)

for file in merged_files:
    file_path = file.path
    print(f"🔹 Processing file: {file_path}")

    # Read file
    df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(file_path)

    # Rename columns: Remove location name and apply clean headers
    new_columns = ["Date"] + [column_rename_map[col_name.split("_")[-1]] for col_name in df.columns[1:]]

    # Apply renaming
    df_cleaned = df.toDF(*new_columns)

    # Save back to CSV
    df_cleaned.write.format("csv").mode("overwrite").option("header", "true").save(file_path)

    print(f"✅ Renamed and saved: {file_path}")

print("🎯 All merged tables have clean headers!")


In [0]:
# Example: Read the "BATALHA.csv" file (replace with any filename you want to check)
file_to_check = "/mnt/weather_data/split_stations/merged/ALCACOVAS.csv"

df_check = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(file_to_check)

# Display the contents of the file
df_check.display()

In [0]:
# Define base directory
merged_dir = "/mnt/weather_data/split_stations/merged/"

# Get list of merged files
merged_files = dbutils.fs.ls(merged_dir)

for file in merged_files:
    file_path = file.path
    print(f"🔹 Processing file: {file_path}")

    # Read file with headers
    df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(file_path)

    # Get the column names before removing rows
    columns = df.columns

    # Remove first two rows (skip metadata but keep headers)
    df_cleaned = df.rdd.zipWithIndex().filter(lambda row: row[1] > 1).map(lambda row: row[0]).toDF(df.schema)

    # Apply the correct column names again
    df_cleaned = df_cleaned.toDF(*columns)

    # Save back to CSV with headers
    df_cleaned.write.format("csv").mode("overwrite").option("header", "true").save(file_path)

    print(f"✅ Cleaned and saved: {file_path}")

print("🎯 All merged tables now have the first two rows removed, but headers remain intact!")


In [0]:
# Example: Read the "BATALHA.csv" file (replace with any filename you want to check)
file_to_check = "/mnt/weather_data/split_stations/merged/BATALHA.csv"

df_check = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(file_to_check)

# Display the contents of the file
df_check.display()

In [0]:
location_coordinates = {
    "ALAGOA/": (39.6833, -7.7667),
    "ALCACOVAS/": (38.3833, -8.0167),
    "BARRAGEM_DE_CASTELO_BURGES/": (39.0833, -9.0833),
    "BARRAGEM_DE_MEIMOA/": (40.1667, -7.1667),
    "BARRAGEM_DO_DIVOR/": (38.7167, -7.8833),
    "BARRAGEM_DO_ROXO/": (37.8833, -8.05),
    "BATALHA/": (39.6608, -8.8258),
    "CAMPO_EXPERIMENTAL_CRATO/": (39.2833, -7.65),
    "CAXARIAS/": (39.6833, -8.5),
    "COLARES/": (38.8, -9.45),
    "COMPORTA/": (38.3833, -8.7833),
    "DEILO/": (41.8667, -6.9333),
    "GONDIZALVES/": (41.5667, -8.4167),
    "JUNQUEIRA/": (41.5333, -8.6),
    "MINAS_DE_JALES/": (41.5167, -7.5),
    "PONTE_DA_BARCA/": (41.8, -8.4167),
    "PROENCA_A_NOVA/": (39.75, -7.9167),
    "SANTAREM/": (39.2333, -8.6833),
    "SAO_BRAS_DE_ALPORTEL/": (37.15, -7.8833)
}


In [0]:
from pyspark.sql.functions import lit

# Define base directory
merged_dir = "/mnt/weather_data/split_stations/merged/"

# Get list of merged files
merged_files = dbutils.fs.ls(merged_dir)

for file in merged_files:
    file_path = file.path
    location_name = file.name.replace(".csv", "")  # Extract location name from filename

    # Check if we have coordinates for this location
    if location_name in location_coordinates:
        latitude, longitude = location_coordinates[location_name]

        print(f"🔹 Adding coordinates to {location_name}...")

        # Read dataset
        df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(file_path)

        # Add Latitude and Longitude columns
        df_with_coords = df.withColumn("Latitude", lit(latitude)).withColumn("Longitude", lit(longitude))

        # Save updated dataset
        df_with_coords.write.format("csv").mode("overwrite").option("header", "true").save(file_path)

        print(f"✅ Updated and saved: {file_path}")
    else:
        print(f"⚠️ No coordinates found for {location_name}, skipping.")

print("🎯 All datasets now include coordinates!")


In [0]:
df_check = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("/mnt/weather_data/split_stations/merged/SANTAREM.csv")
df_check.display()


In [0]:
# Function to clean column names for Delta tables
def clean_column_name(name):
    return (
        name.lower()  # Convert to lowercase
        .replace(" ", "_")  # Replace spaces with underscores
        .replace("(", "")  # Remove parentheses
        .replace(")", "")  # Remove parentheses
        .replace("%", "percent")  # Convert % to "percent"
        .replace(",", "")  # Remove commas
        .replace(";", "")  # Remove semicolons
    )

# Define base directories
merged_dir = "/mnt/weather_data/split_stations/merged/"
delta_base_path = "/mnt/weather_data/delta/"

# Get list of merged files
merged_files = dbutils.fs.ls(merged_dir)

for file in merged_files:
    file_path = file.path
    location_name = file.name.replace(".csv", "")  # Extract location name

    print(f"🔹 Converting {location_name} to Delta format...")

    # Read CSV
    df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(file_path)

    # Clean column names
    new_columns = [clean_column_name(col_name) for col_name in df.columns]
    df_cleaned = df.toDF(*new_columns)

    # Define Delta table path
    delta_path = f"{delta_base_path}{location_name}"

    # Write to Delta format
    df_cleaned.write.format("delta").option("mergeSchema", "true").mode("overwrite").save(delta_path)

    print(f"✅ Delta table saved at: {delta_path}")

print("🎯 All datasets are now saved in Delta format with cleaned column names!")


In [0]:
display(dbutils.fs.ls("/mnt/weather_data/delta/"))

In [0]:
df_check = spark.read.format("delta").option("header", "true").option("inferSchema", "true").load("/mnt/weather_data/delta/SANTAREM/")
df_check.display()


In [0]:
import re

# Function to clean table names for Hive Metastore
def clean_table_name(name):
    name = name.rstrip("/")  # Remove trailing slashes
    name = re.sub(r"[^a-zA-Z0-9_]", "_", name)  # Replace invalid characters with "_"
    return name.lower()  # Convert to lowercase for consistency

# Define Hive table name (adjust schema/database if needed)
for file in dbutils.fs.ls(delta_base_path):
    location_name = clean_table_name(file.name)  # Clean table name

    print(f"🔹 Saving as Hive table: {location_name}...")

    # Read Delta table
    df = spark.read.format("delta").load(f"{delta_base_path}{file.name}")

    # Save as Hive table
    df.write.format("delta").mode("overwrite").saveAsTable(location_name)

    print(f"✅ Hive table created: {location_name}")

print("🎯 All Delta tables are now registered in the Hive Metastore!")


In [0]:
%sql
CREATE TABLE weather_data.alagoa AS SELECT * FROM default.alagoa;
DROP TABLE default.alagoa;

CREATE TABLE weather_data.alcacovas AS SELECT * FROM default.alcacovas;
DROP TABLE default.alcacovas;

CREATE TABLE weather_data.barragem_de_castelo_burgoes AS SELECT * FROM default.barragem_de_castelo_burges;
DROP TABLE default.barragem_de_castelo_burges;

CREATE TABLE weather_data.barragem_de_meimoa AS SELECT * FROM default.barragem_de_meimoa;
DROP TABLE default.barragem_de_meimoa;

CREATE TABLE weather_data.barragem_do_divor AS SELECT * FROM default.barragem_do_divor;
DROP TABLE default.barragem_do_divor;

CREATE TABLE weather_data.barragem_do_roxo AS SELECT * FROM default.barragem_do_roxo;
DROP TABLE default.barragem_do_roxo;

CREATE TABLE weather_data.batalha AS SELECT * FROM default.batalha;
DROP TABLE default.batalha;

CREATE TABLE weather_data.campo_experimental_crato AS SELECT * FROM default.campo_experimental_crato;
DROP TABLE default.campo_experimental_crato;

CREATE TABLE weather_data.caxarias AS SELECT * FROM default.caxarias;
DROP TABLE default.caxarias;

CREATE TABLE weather_data.colares AS SELECT * FROM default.colares;
DROP TABLE default.colares;

CREATE TABLE weather_data.comporta AS SELECT * FROM default.comporta;
DROP TABLE default.comporta;

CREATE TABLE weather_data.deilao AS SELECT * FROM default.deilo;
DROP TABLE default.deilo;

CREATE TABLE weather_data.gondizalves AS SELECT * FROM default.gondizalves;
DROP TABLE default.gondizalves;

CREATE TABLE weather_data.junqueira AS SELECT * FROM default.junqueira;
DROP TABLE default.junqueira;

CREATE TABLE weather_data.minas_de_jales AS SELECT * FROM default.minas_de_jales;
DROP TABLE default.minas_de_jales;

CREATE TABLE weather_data.ponte_da_barca AS SELECT * FROM default.ponte_da_barca;
DROP TABLE default.ponte_da_barca;

CREATE TABLE weather_data.proenca_a_nova AS SELECT * FROM default.proenca_a_nova;
DROP TABLE default.proenca_a_nova;

CREATE TABLE weather_data.santarem AS SELECT * FROM default.santarem;
DROP TABLE default.santarem;

CREATE TABLE weather_data.sao_bras_de_alportel AS SELECT * FROM default.sao_bras_de_alportel;
DROP TABLE default.sao_bras_de_alportel;



In [0]:
%sql
CREATE DATABASE IF NOT EXISTS weather_data;